# Quick start

This is a fast tour of linopy's core moves: declare variables, write constraints, set an objective, solve. We start with a scalar model so the syntax is bare, then add a coordinate so you see how linopy scales to real data.

## A scalar model

$$
\begin{aligned}
\min_{x, y} \quad & x + 2y \\
\text{s.t.} \quad & 3x + 7y \ge 10 \\
& 5x + 2y \ge 3 \\
& x, y \ge 0
\end{aligned}
$$

In [ ]:
import linopy

m = linopy.Model()

Each call to `add_variables` returns a `Variable` object. With no coordinates, it represents a single scalar decision.

In [ ]:
x = m.add_variables(lower=0, name="x")
y = m.add_variables(lower=0, name="y")
x

Constraints are written in natural mathematical notation. linopy figures out which side has the variables and which has the constants — the two forms below are equivalent.

In [ ]:
3 * x + 7 * y >= 10

In [ ]:
m.add_constraints(3 * x + 7 * y >= 10, name="c1")
m.add_constraints(5 * x + 2 * y >= 3, name="c2");

Set the objective and solve.

In [ ]:
m.add_objective(x + 2 * y)
m.solve(solver_name="highs", output_flag=False)

In [ ]:
float(x.solution), float(y.solution)

## Adding coordinates

Real optimization problems have data: prices indexed by time, capacities indexed by asset, demand indexed by region. linopy variables can carry the same named coordinates as your `pandas` or `xarray` data — the syntax doesn't change.

Let's parameterise the right-hand side by a time index $t \in \{0, \dots, 9\}$:

$$
\begin{aligned}
\min \quad & \sum_t (x_t + 2 y_t) \\
\text{s.t.} \quad & 3 x_t + 7 y_t \ge 10 t & \forall t \\
& 5 x_t + 2 y_t \ge 3 t & \forall t \\
& x_t, y_t \ge 0 & \forall t
\end{aligned}
$$

In [ ]:
import pandas as pd

time = pd.Index(range(10), name="time")

m = linopy.Model()
x = m.add_variables(lower=0, coords=[time], name="x")
y = m.add_variables(lower=0, coords=[time], name="y")
x

`x` and `y` now hold ten variables each, labelled by `time`. Constraints and objectives written with them broadcast over that coordinate automatically — `t` on the right-hand side aligns with the `time` dimension on the left.

The payoff: because variables share coordinates with your input data, scaling from ten timesteps to ten thousand is just a matter of feeding larger inputs. The model code stays the same shape — no loops to grow, no indexers to rewrite.

In [ ]:
t = pd.Series(time, index=time)

m.add_constraints(3 * x + 7 * y >= 10 * t, name="c1")
m.add_constraints(5 * x + 2 * y >= 3 * t, name="c2")

m.add_objective((x + 2 * y).sum())
m.solve(solver_name="highs", output_flag=False);

In [ ]:
m.solution.to_dataframe().plot(grid=True, ylabel="value", figsize=(6, 3));

## Where to next

- [Your first real model](first-real-model.ipynb) walks through a small economic-dispatch problem with two dimensions and real input data.
- The **Creating a Model** section covers variables, expressions, constraints, and coordinate alignment in depth.
- [Solving a model](solving.ipynb) covers choosing a solver and inspecting it afterwards; [The Solver API](solver-api.ipynb) is the lower-level construct-then-solve interface.